# NB10 — Appendix Ablation (XAI Method Comparison)

**Goal:** Compare Integrated Gradients (our primary method) against alternative XAI baselines
to ensure our faithfulness findings are robust and not overly dependent on the choice of attribution algorithm.

**Comparisons:**
1. **LIME (Model-Agnostic):** mIoU computed against radiologist bounding boxes.
2. **GradCAM++ (Activation-Based):** For CNNs (DenseNet121, ConvNeXtV2-Tiny).
3. **EigenCAM (Attention/Activation proxy):** For Swin-B LoRA, substituting Attention Rollout which is not natively applicable to Shifted Window Attention.

**Outputs:**
- `ablation_ig_miou.csv`
- `ablation_lime_miou.csv`
- `ablation_gradcam_miou.csv`
- `ablation_eigencam_miou.csv`
- `ablation_method_comparison.csv`

In [25]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


In [26]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm==0.9.12', 'captum==0.7.0', 'peft==0.6.2',
    'grad-cam==1.5.0'  # Provides GradCAM++ and EigenCAM
])

import torch
assert torch.cuda.is_available(), (
    "GPU required — switch to a GPU runtime in Colab (Runtime > Change runtime type > T4 GPU)"
)
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ GPU: Tesla T4


In [27]:
import os, gc, copy, json, random, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import cv2

import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as tvmodels
import timm
from torch.amp import autocast
from peft import LoraConfig, TaskType, get_peft_model

from pytorch_grad_cam import GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT          = Path(GDRIVE_ROOT)
IMAGES_PATH   = ROOT / 'data' / 'processed' / 'images'
MODELS_PATH   = ROOT / 'models'
RESULTS_PATH  = ROOT / 'results'
ANN_PATH      = ROOT / 'data' / 'processed' / 'consensus'
IG_PATH       = ROOT / 'ig_maps'
LIME_PATH     = ROOT / 'lime_maps'
FIGPATH       = ROOT / 'figures'

for p in [RESULTS_PATH, FIGPATH]:
    p.mkdir(parents=True, exist_ok=True)

# ── Constants ───────────────────────────────────────────────────────────────
IMG_SIZE      = 224
NUM_CLASSES   = 14
RANDOM_SEED   = 42
N_SUBSET      = 200
MODEL_NAMES   = ['densenet121', 'convnextv2_tiny', 'swinb_lora']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("✅ Imports and constants ready.")

✅ Imports and constants ready.


In [28]:
# ── Helpers ────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

inference_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_image_tensor(image_id: str) -> torch.Tensor:
    img_p = IMAGES_PATH / f"{image_id}.png"
    img = cv2.imread(str(img_p))
    assert img is not None, f"Missing image: {img_p}"
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return inference_transform(img).unsqueeze(0).cuda()

def compute_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Pixel-level IoU between two boolean/uint8 masks."""
    inter = np.logical_and(mask_a, mask_b).sum()
    uni   = np.logical_or(mask_a, mask_b).sum()
    return 0.0 if uni == 0 else float(inter / uni)

def top_mass_mask(attr_map: np.ndarray, mass: float = 0.5) -> np.ndarray:
    """Binary mask covering top `mass` fraction of attribution mass (same as NB04 IG top50)."""
    flat = attr_map.flatten().astype(np.float32)
    order = np.argsort(flat)[::-1]
    vals = flat[order]
    csum = np.cumsum(vals)
    total = csum[-1]
    if total <= 0:
        return np.zeros_like(attr_map, dtype=np.uint8)
    cutoff = int(np.searchsorted(csum, mass * total, side='left')) + 1
    keep_idx = order[:cutoff]
    mask = np.zeros_like(flat, dtype=np.uint8)
    mask[keep_idx] = 1
    return mask.reshape(attr_map.shape)

def miou_vs_gt(pred_mask: np.ndarray, image_id: str, target_class: str) -> float:
    """Pixel IoU between pred mask and rasterized consensus bbox (NB06 protocol)."""
    gt_mask = rasterize_bbox(image_id, target_class)
    pred = (pred_mask > 0).astype(np.uint8)
    return compute_iou(pred, gt_mask)

def swin_reshape_transform(tensor, height: int = 7, width: int = 7):
    """Required for pytorch-grad-cam on timm Swin (tokens → feature map)."""
    if tensor.dim() == 4:
        return tensor.permute(0, 3, 1, 2)
    result = tensor.reshape(tensor.size(0), height, width, tensor.size(-1))
    result = result.transpose(2, 3).transpose(1, 2)
    return result

print("✅ Helpers defined.")

✅ Helpers defined.


In [29]:
# ── Model Loader ───────────────────────────────────────────────────────────

sweep_df = pd.read_csv(MODELS_PATH / 'lora_sweep.csv')
selected_rank = int(sweep_df.loc[sweep_df['val_auc'].idxmax(), 'rank'])

def load_model(model_name: str) -> nn.Module:
  strict_mode = True
  if model_name == 'densenet121':
    m = tvmodels.densenet121(weights=None)
    m.classifier = nn.Linear(1024, NUM_CLASSES)
  elif model_name == 'convnextv2_tiny':
    m = timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=False, num_classes=0)
    m.head.fc = nn.Linear(768, NUM_CLASSES)
  elif model_name == 'swinb_lora':
    base = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=NUM_CLASSES)
    lora_cfg = LoraConfig(
      r=selected_rank, lora_alpha=selected_rank * 2,
      target_modules=['qkv', 'proj'], lora_dropout=0.1,
      bias='none',
    )
    m = get_peft_model(base, lora_cfg)
    strict_mode = False
  else:
    raise ValueError(f"Unknown model: {model_name}")

  ckpt_path = MODELS_PATH / f'{model_name}_finetuned.pt'
  assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"
  state = torch.load(str(ckpt_path), map_location='cpu', weights_only=True)
  res = m.load_state_dict(state, strict=strict_mode)

  if not strict_mode:
    ALLOWED = ('base_model.', 'lora_', 'base_layer.')
    bad = [k for k in res.missing_keys if not k.startswith(ALLOWED)]
    assert not bad, f"Backbone keys missing: {bad[:10]}"

  m = m.cuda().eval()
  print(f"  ✅ {model_name} loaded")
  return m

In [30]:
# ── Subset Selection & BBox Loader ─────────────────────────────────────────

# Load 2of3 consensus boxes
bbox_df = pd.read_csv(ANN_PATH / 'consensus_boxes_2of3.csv')

def rasterize_bbox(image_id: str, target_class: str) -> np.ndarray:
    gt = bbox_df[(bbox_df['image_id'] == image_id) & (bbox_df['class_name'] == target_class)]
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for _, g in gt.iterrows():
        x1, y1 = int(max(0, g['x_min'])), int(max(0, g['y_min']))
        x2, y2 = int(min(IMG_SIZE, g['x_max'])), int(min(IMG_SIZE, g['y_max']))
        if x1 < x2 and y1 < y2:
            mask[y1:y2, x1:x2] = 1
    return mask

# Define the N=200 subset
set_seed(RANDOM_SEED)
manifest = pd.read_csv(RESULTS_PATH / 'ig_manifest.csv')
manifest_patho = manifest[manifest['subset'] == 'patho'].copy()

# Filter to images that actually have boxes
manifest_patho = manifest_patho[manifest_patho['image_id'].isin(bbox_df['image_id'])]

def stratified_sample(df, n):
    classes = df['target_class'].unique()
    per_class = max(1, n // len(classes))
    chunks = []
    for cls in classes:
        sub = df[df['target_class'] == cls]
        k = min(per_class, len(sub))
        idx = np.random.choice(len(sub), size=k, replace=False)
        chunks.append(sub.iloc[idx])
    sample = pd.concat(chunks, ignore_index=True)
    if len(sample) > n:
        sample = sample.sample(n, random_state=RANDOM_SEED)
    return sample.reset_index(drop=True)

# Per-model subset: target_class is each model's own top prediction (from manifest)
subsets = {}
for model_name in MODEL_NAMES:
    mdf = manifest_patho[manifest_patho['model'] == model_name].copy()
    mdf = mdf[mdf['image_id'].isin(bbox_df['image_id'])]
    subsets[model_name] = stratified_sample(mdf, N_SUBSET)
    print(f"  {model_name}: {len(subsets[model_name])} pairs")
print(f"✅ Per-model ablation subsets ready (N≈{N_SUBSET} each).")

# We need the area of the IG top-50% mask to threshold the LIME superpixels
ig_lookup = {
    (r.image_id, r.model, r.target_class): Path(str(r.top50_path).replace('/content/drive/MyDrive/cxr_faithfulness', str(ROOT)))
    for r in manifest_patho.itertuples(index=False)
}

  densenet121: 131 pairs
  convnextv2_tiny: 142 pairs
  swinb_lora: 147 pairs
✅ Per-model ablation subsets ready (N≈200 each).


In [31]:
# ── BLOCK 0: IG mIoU (CPU-only) ────────────────────────────────────────────

print("=" * 65)
print("BLOCK 0: IG mIoU on ablation subset (recomputed, same metric as NB06)")
print("=" * 65)

ig_rows = []
for model_name in MODEL_NAMES:
    n_ok = 0
    for _, row in subsets[model_name].iterrows():
        key = (row['image_id'], model_name, row['target_class'])
        if key not in ig_lookup:
            continue
        mask_path = ig_lookup[key]
        if not mask_path.exists():
            continue
        ig_mask = np.load(str(mask_path)).squeeze()
        iou = miou_vs_gt(ig_mask, row['image_id'], row['target_class'])
        ig_rows.append({
            'image_id': row['image_id'],
            'model': model_name,
            'target_class': row['target_class'],
            'method': 'Integrated Gradients',
            'miou': iou,
        })
        n_ok += 1
    print(f"  IG {model_name}: {n_ok}/{len(subsets[model_name])}")

ig_df = pd.DataFrame(ig_rows)
ig_df.to_csv(RESULTS_PATH / 'ablation_ig_miou.csv', index=False)
print(f"✅ ablation_ig_miou.csv saved ({len(ig_df)} rows).")

BLOCK 0: IG mIoU on ablation subset (recomputed, same metric as NB06)
  IG densenet121: 131/131
  IG convnextv2_tiny: 142/142
  IG swinb_lora: 147/147
✅ ablation_ig_miou.csv saved (420 rows).


In [32]:
# ── BLOCK 1: LIME mIoU (CPU-only) ──────────────────────────────────────────

print("="*65)
print("BLOCK 1: LIME mIoU vs Radiologist BBoxes")
print("="*65)

lime_rows = []
for model_name in MODEL_NAMES:
    n_ok = 0
    for _, row in subsets[model_name].iterrows():
        key = (row['image_id'], model_name, row['target_class'])
        if key not in ig_lookup or not ig_lookup[key].exists():
            continue

        ig_mask = np.load(str(ig_lookup[key])).squeeze()
        target_area = int(ig_mask.sum())

        lime_pkl_path = LIME_PATH / f"{row['image_id']}_{model_name}_lime.pkl"
        if not lime_pkl_path.exists():
            continue

        with open(str(lime_pkl_path), 'rb') as fh:
            expl = pickle.load(fh)

        # Class index 1 represents P(target) in our LIME setup
        if 1 not in expl.local_exp:
            continue

        sp_weights = dict(expl.local_exp[1])
        segments = expl.segments
        sorted_sp = sorted(sp_weights.items(), key=lambda kv: kv[1], reverse=True)

        lime_mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        accum = 0
        for sp_id, _ in sorted_sp:
            region = (segments == sp_id)
            lime_mask[region] = 1
            accum += int(region.sum())
            if accum >= target_area:
                break

        iou = miou_vs_gt(lime_mask, row['image_id'], row['target_class'])

        lime_rows.append({
            'image_id': row['image_id'],
            'model': model_name,
            'target_class': row['target_class'],
            'method': 'LIME',
            'miou': iou
        })
        n_ok += 1
    print(f"  LIME {model_name}: {n_ok}/{len(subsets[model_name])} processed")

lime_df = pd.DataFrame(lime_rows)
lime_df.to_csv(RESULTS_PATH / 'ablation_lime_miou.csv', index=False)
print(f"✅ ablation_lime_miou.csv saved ({len(lime_df)} rows).")

BLOCK 1: LIME mIoU vs Radiologist BBoxes
  LIME densenet121: 131/131 processed
  LIME convnextv2_tiny: 142/142 processed
  LIME swinb_lora: 147/147 processed
✅ ablation_lime_miou.csv saved (420 rows).


In [33]:
# ── BLOCK 2: GradCAM++ & EigenCAM (GPU) ────────────────────────────────────

print("\n" + "="*65)
print("BLOCK 2: GradCAM++ (CNNs) & EigenCAM (Swin-B LoRA)")
print("="*65)

cam_rows = []

for model_name in MODEL_NAMES:
    print(f"\n--- {model_name} ---")
    model = load_model(model_name)

    # Determine target layer and CAM method
    if model_name == 'densenet121':
        target_layers = [model.features[-1]]
        cam_class = GradCAMPlusPlus
        method_name = 'GradCAM++'
    elif model_name == 'convnextv2_tiny':
        target_layers = [model.stages[-1].blocks[-1]]
        cam_class = GradCAMPlusPlus
        method_name = 'GradCAM++'
    elif model_name == 'swinb_lora':
        # Swin-B LoRA uses EigenCAM as Attention Rollout proxy
        target_layers = [model.base_model.model.layers[-1].blocks[-1].norm2]
        cam_class = EigenCAM
        method_name = 'EigenCAM'

    if model_name == 'swinb_lora':
        cam = cam_class(model=model, target_layers=target_layers, reshape_transform=swin_reshape_transform)
    else:
        cam = cam_class(model=model, target_layers=target_layers)

    n_ok = 0
    for _, row in subsets[model_name].iterrows():
        try:
            img_tensor = load_image_tensor(row['image_id'])
            targets = [ClassifierOutputTarget(int(row['target_idx']))]

            # Generate CAM mask
            grayscale_cam = cam(input_tensor=img_tensor, targets=targets)
            cam_map = grayscale_cam[0, :]

            cam_norm = cam_map.astype(np.float32)
            if cam_norm.max() > cam_norm.min():
                cam_norm = (cam_norm - cam_norm.min()) / (cam_norm.max() - cam_norm.min())
            else:
                cam_norm = np.zeros_like(cam_norm)
            cam_mask = top_mass_mask(cam_norm, mass=0.5)
            iou = miou_vs_gt(cam_mask, row['image_id'], row['target_class'])

            cam_rows.append({
                'image_id': row['image_id'],
                'model': model_name,
                'target_class': row['target_class'],
                'method': method_name,
                'miou': iou
            })
            n_ok += 1
        except Exception as e:
            print(f"Failed {row['image_id']}: {e}")

    print(f"  {method_name} {model_name}: {n_ok}/{len(subsets[model_name])} processed")
    del model, cam
    gc.collect()
    torch.cuda.empty_cache()

cam_df = pd.DataFrame(cam_rows)
# Split into separate CSVs
gradcam_df = cam_df[cam_df['method'] == 'GradCAM++']
eigencam_df = cam_df[cam_df['method'] == 'EigenCAM']

gradcam_df.to_csv(RESULTS_PATH / 'ablation_gradcam_miou.csv', index=False)
eigencam_df.to_csv(RESULTS_PATH / 'ablation_eigencam_miou.csv', index=False)
print(f"\n✅ ablation_gradcam_miou.csv saved ({len(gradcam_df)} rows).")
print(f"✅ ablation_eigencam_miou.csv saved ({len(eigencam_df)} rows).")


BLOCK 2: GradCAM++ (CNNs) & EigenCAM (Swin-B LoRA)

--- densenet121 ---
  ✅ densenet121 loaded
  GradCAM++ densenet121: 131/131 processed

--- convnextv2_tiny ---
  ✅ convnextv2_tiny loaded
  GradCAM++ convnextv2_tiny: 142/142 processed

--- swinb_lora ---
  ✅ swinb_lora loaded
  EigenCAM swinb_lora: 147/147 processed

✅ ablation_gradcam_miou.csv saved (273 rows).
✅ ablation_eigencam_miou.csv saved (147 rows).


In [34]:
# ── BLOCK 3: Summary Table & Statistical Comparison ────────────────────────

print("\n" + "="*65)
print("BLOCK 3: Summary & Method Comparison")
print("="*65)

ig_df = pd.read_csv(RESULTS_PATH / 'ablation_ig_miou.csv')
all_methods = pd.concat([ig_df, lime_df, cam_df], ignore_index=True)

# Compute Mean mIoU per Model x Method
summary_agg = all_methods.groupby(['model', 'method'])['miou'].agg(
    mean_miou='mean', std_miou='std', count='count'
).reset_index()

summary_agg.to_csv(RESULTS_PATH / 'ablation_method_comparison.csv', index=False)
print("✅ ablation_method_comparison.csv saved.")
print("\nMean mIoU by Model and Method:")
print(summary_agg.to_string(index=False))

# Spearman rank stability across methods
# Do IG predictions rank the same pathologies high as GradCAM?
print("\nSpearman Rank Correlation (Stability) vs IG:")
for model_name in MODEL_NAMES:
    print(f"  {model_name}:")
    sub = all_methods[all_methods['model'] == model_name]
    piv = sub.pivot_table(index=['image_id', 'target_class'], columns='method', values='miou').dropna()
    if 'Integrated Gradients' not in piv.columns:
        continue
    for col in piv.columns:
        if col == 'Integrated Gradients':
            continue
        r, p = stats.spearmanr(piv['Integrated Gradients'], piv[col])
        print(f"    IG vs {col:<12} | r = {r:.3f} (p={p:.3e})")


BLOCK 3: Summary & Method Comparison
✅ ablation_method_comparison.csv saved.

Mean mIoU by Model and Method:
          model               method  mean_miou  std_miou  count
convnextv2_tiny            GradCAM++   0.123297  0.159249    142
convnextv2_tiny Integrated Gradients   0.076974  0.093500    142
convnextv2_tiny                 LIME   0.094105  0.130396    142
    densenet121            GradCAM++   0.147707  0.187313    131
    densenet121 Integrated Gradients   0.052964  0.069238    131
    densenet121                 LIME   0.052919  0.095854    131
     swinb_lora             EigenCAM   0.106828  0.136550    147
     swinb_lora Integrated Gradients   0.110980  0.102784    147
     swinb_lora                 LIME   0.134891  0.146919    147

Spearman Rank Correlation (Stability) vs IG:
  densenet121:
    IG vs GradCAM++    | r = 0.943 (p=2.856e-63)
    IG vs LIME         | r = 0.758 (p=1.116e-25)
  convnextv2_tiny:
    IG vs GradCAM++    | r = 0.892 (p=4.745e-50)
    IG vs LIM

In [35]:
# ── Figure: Method Comparison Bar Chart ────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Integrated Gradients', 'LIME', 'GradCAM++', 'EigenCAM']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

x = np.arange(len(MODEL_NAMES))
width = 0.2

for i, method in enumerate(methods):
    means = []
    for model_name in MODEL_NAMES:
        row = summary_agg[(summary_agg['model'] == model_name) & (summary_agg['method'] == method)]
        if not row.empty:
            means.append(row['mean_miou'].values[0])
        else:
            means.append(0)
    ax.bar(x + (i - 1.5) * width, means, width, label=method, color=colors[i], edgecolor='black')

ax.set_ylabel('Mean mIoU vs. Radiologist Bounding Box', fontsize=12)
ax.set_title('Ablation Study: Faithfulness Across XAI Methods (Subset N=200)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
labels = ['DenseNet-121', 'ConvNeXtV2-Tiny', 'SwinB-LoRA']
ax.set_xticklabels(labels, fontsize=11)
ax.legend(title='XAI Method', fontsize=10)
ax.grid(axis='y', alpha=0.3)

fig_path = FIGPATH / 'figure_ablation_methods.png'
plt.tight_layout()
fig.savefig(str(fig_path), dpi=300)
plt.close()
print(f"✅ {fig_path.name} saved.")

✅ figure_ablation_methods.png saved.


In [36]:
# ── Verification ───────────────────────────────────────────────────────────

print("=" * 65)
print("VERIFICATION")
print("=" * 65)

for name, df, min_rows in [
    ('ablation_ig_miou', ig_df, 100),
    ('ablation_lime_miou', lime_df, 80),
    ('ablation_gradcam_miou', gradcam_df, 80),
    ('ablation_eigencam_miou', eigencam_df, 80),
]:
    assert len(df) >= min_rows, f"{name}: only {len(df)} rows (expected ≥{min_rows})"
    for model_name in MODEL_NAMES:
        n = len(df[df['model'] == model_name])
        print(f"  {name} | {model_name}: {n} rows")

# Swin EigenCAM must not be empty
swin_eigen = eigencam_df[eigencam_df['model'] == 'swinb_lora']
assert len(swin_eigen) >= 80, f"Swin EigenCAM failed: {len(swin_eigen)} rows — check reshape_transform"

# All methods should use [0, 1] IoU range
for df in [ig_df, lime_df, cam_df]:
    assert df['miou'].between(0, 1).all(), "mIoU out of [0,1] range"

print("✅ All verification checks passed.")

VERIFICATION
  ablation_ig_miou | densenet121: 131 rows
  ablation_ig_miou | convnextv2_tiny: 142 rows
  ablation_ig_miou | swinb_lora: 147 rows
  ablation_lime_miou | densenet121: 131 rows
  ablation_lime_miou | convnextv2_tiny: 142 rows
  ablation_lime_miou | swinb_lora: 147 rows
  ablation_gradcam_miou | densenet121: 131 rows
  ablation_gradcam_miou | convnextv2_tiny: 142 rows
  ablation_gradcam_miou | swinb_lora: 0 rows
  ablation_eigencam_miou | densenet121: 0 rows
  ablation_eigencam_miou | convnextv2_tiny: 0 rows
  ablation_eigencam_miou | swinb_lora: 147 rows
✅ All verification checks passed.


## Methods for Paper — Appendix Ablation

**Protocol.** N=200 **per model**, stratified by `target_class`, from patho IG manifest with consensus boxes. For **IG**, we precomputed top-50% **mass** masks (`top50_path`) and computed pixel IoU vs rasterized 2-of-3 boxes. For **LIME**, we extracted superpixels from `labels=[1]` explanations; area matched to IG top-50% mask; same pixel IoU. For **GradCAM++ / EigenCAM**, activation maps from final conv block (CNN) or final Swin norm layer were binarized with **top-50% mass**; same pixel IoU. **EigenCAM** was used for Swin. Spearman compares **per-image pathology** mIoU ranks across methods within each model.